In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### **Read Data from Bronze Layer**

In [0]:
bronze_df = spark.read.table("fmcg.bronze.customers")
# bronze_df = spark.sql("select * from fmcg.bronze.customers")

### **Importing Prerequisites**

In [0]:
%run "/Workspace/Users/0bcr9999@gmail.com/FMGC-DATAENGINEERING-PROJECT/notebooks/setup/3. utilities"

In [0]:
dbutils.widgets.text("catalog", "fmcg")
dbutils.widgets.text("data_source", "", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

print(catalog, data_source)

### **Duplicate Removal From Data**

In [0]:
df_duplicates = bronze_df.groupBy("customer_id").count().filter("count > 1")
display(df_duplicates)


In [0]:
print("count of rows before droping duplicates : ", bronze_df.count())
bronze_uniq_df = bronze_df.dropDuplicates(["customer_id"])
print("count of rows after droping duplicates : ", bronze_uniq_df.count())

### **Cleaning of customer_name - Removal Unwanted Spaces**

In [0]:
display(
    bronze_uniq_df.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
bronze_trim_df = bronze_uniq_df.withColumn("customer_name", F.trim(F.col("customer_name")))

In [0]:
display(
    bronze_trim_df.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

### **Cleaning of customer_name - Text Formating by handling Fat Fingering**

In [0]:
display(bronze_trim_df.select("customer_name").distinct())

In [0]:
bronze_case_df = bronze_trim_df \
    .withColumn("customer_name", 
                F.when(F.col("customer_name").isNull(), None)
                .otherwise(F.initcap(F.col("customer_name")))
                )

In [0]:
display(bronze_case_df.select("customer_name").distinct())

### **Cleaning of city - Text Formating by handling Fat Fingering**

In [0]:
bronze_case_df.select("city").distinct().display()

In [0]:
city_mapping = {
    "Bengaluru" : "Bangalore",
    "Bengalore" : "Bangalore",
    "Bengaluruu" : "Bangalore",

    "Hyderabad" : "Hyderabad",
    "Hyderabadd" : "Hyderabad",
    "Hyderbad" : "Hyderabad",

    "New Delhi" : "New Delhi",
    "NewDelhee" : "New Delhi",
    "NewDelhi" : "New Delhi",
    "NewDheli" : "New Delhi"
}

allowed = ["Bangalore", "Hyderabad", "New Delhi"]

bronze_fatfinger_df = bronze_case_df.replace(city_mapping, subset = ["city"])
bronze_ff_df = bronze_fatfinger_df \
    .withColumn("city", 
                F.when(F.col("city").isNull(), None)
                .when(F.col("city").isin(allowed), F.col("city"))
                .otherwise(None)
                )
    
# Sanity Check
display(bronze_ff_df.select("city").distinct().display())

### **Cleaning of city - Null Handling**

In [0]:
bronze_ff_df.filter(F.col("city").isNull()).display()

In [0]:
null_customer_df = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']

bronze_ff_df.filter(F.col("customer_name").isin(null_customer_df)).display()

In [0]:
customer_fix = {
    "789403" : "New Delhi",
    "789420" : "Bangalore",
    "789521" : "Hyderabad",
    "789603" : "Hyderabad"
}
data = []

for k, v in customer_fix.items():
        data.append((k, v))

customer_fix_df = spark.createDataFrame(data , ["customer_id", "fixed_city"])

In [0]:
display(customer_fix_df)

In [0]:
bronze_null_df = bronze_ff_df.join(customer_fix_df, "customer_id", "left") \
    .withColumn("city", 
                F.coalesce(F.col("city"), F.col("fixed_city"))) \
                    .drop("fixed_city")
    
display(bronze_null_df)


### **Column Standardization: Combining customer_name and city into a Unified Field**

In [0]:
bronze_clean_df = bronze_null_df.withColumn("customer", F.concat(F.col("customer_name"), F.lit("-"), F.coalesce(F.col("city"), F.lit("Unknown"))))

display(bronze_clean_df)

### **Addition of Static Attributes as per Parent Data Model**

In [0]:
bronze_final_df = bronze_clean_df \
  .withColumn("market", F.lit("India")) \
    .withColumn("platform", F.lit("Sports Bar")) \
      .withColumn("channel", F.lit("Acquisition"))

In [0]:
display(bronze_final_df)

### **Write the Final Data Frame to Silver Schema**

In [0]:
bronze_final_df \
  .write \
    .format("delta") \
      .option("delta.enableChangeDataFeed", True) \
        .option("mergeSchema", True) \
          .mode("overwrite") \
            .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")
